In [28]:
import os
import sys

# Pfad zum übergeordneten Verzeichnis hinzufügen (damit themealdb_client gefunden wird)
sys.path.insert(0, os.path.abspath('..'))

from themealdb_client import TheMealDBClient
import json
import traceback

# Test für get_all_ingredients() Funktion

# Client initialisieren
client = TheMealDBClient()
def get_all_ingredients():
    try:
        ingredients = client.get_all_ingredients()
        ingredients = [ingredient['strIngredient'] for ingredient in ingredients if 'strIngredient' in ingredient]
        print(f"Anzahl Zutaten gefunden: {len(ingredients)}")
        print("Beispiel-Zutaten:")
        for ing in ingredients[:10]:  # Zeige die ersten 10 Zutaten
            print(f"- {ing}")
        return ingredients
    except Exception as e:
        print("Fehler beim Abrufen der Zutaten:")
        traceback.print_exc()
        return []
ALL_INGREDIENTS = get_all_ingredients()

# ALL_INGREDIENTS = [
#     "Chicken",
#     "Salmon",
#     "Beef",
#     "Pork",
#     "Avocado",
#     "Apple Cider Vinegar",
#     "Asparagus",
#     "Aubergine",
#     "Baby Plum Tomatoes",
#     "Bacon",
#     "Vinegar",
#     "Tomatoes",
#     "Cucumber"
# ]

Anzahl Zutaten gefunden: 877
Beispiel-Zutaten:
- Chicken
- Salmon
- Beef
- Pork
- Avocado
- Apple Cider Vinegar
- Asparagus
- Aubergine
- Baby Plum Tomatoes
- Bacon


In [38]:
import spacy

# Vektoren funktionieren nur mit md oder lg!
try:
    nlp_score   = spacy.load("en_core_web_md")
    nlp_head    = spacy.load("en_core_web_trf")
except:
    print("Bitte lade das Medium-Modell: python -m spacy download en_core_web_md")
    nlp_score = spacy.load("en_core_web_sm") # Fallback (wird aber Warnung werfen)
    
# 1. NOISE (Ignorieren wir komplett)
NOISE_WORDS = {
    "leaves", "leaf", "seed", "seeds", "nuts", "yolks"
}

MANUAL_CORRECTIONS = {"leaves": "leaf", "leave": "leaf", "nuts": "nut", "yolks": "yolk"}
    
def analyze_grammar(text):
        doc = nlp_head(text.lower())
        
        # 1. Den grammatikalischen Kern finden (ROOT) (tomato sauce -> sauce)
        # Das Wort, das von keinem anderen abhängt
        head_token = [t for t in doc if t.head == t][0]
           
        # 2. Modifiers sammeln (erweiterte information)
        modifiers = [] # Nomen, die den Typ bestimmen (Walnut -> Oil)
        
        for child in head_token.children:
            # # 1. Ignoriere Adjektive (amod) wie "chopped", "fresh", "green"
            # if child.dep_ == "amod":
            #     continue
                
            # # 2. Ignoriere Mengenangaben (nummod) wie "2", "two"
            # elif child.dep_ == "nummod":
            #     continue
                
            # 3. Ignoriere Präpositionen (prep) wie "of" in "Cup of Walnuts"
            if child.dep_ == "prep":
                continue

            else:
                # manuelle Korrektur für modifiers
                modifiers.append(MANUAL_CORRECTIONS.get(child.text, child.lemma_))
                
        # sortiere die Modifier alphabetisch für Konsistenz
        modifiers.sort()
        modifiers_combined = " ".join(modifiers)
        
        # 3. Manuelle Korrekturen für Head (z.B. "leaves" -> "leaf")
        found_head = MANUAL_CORRECTIONS.get(head_token.text, head_token.lemma_)
         
        if found_head in NOISE_WORDS:
            found_head = modifiers_combined
            modifiers_combined = ""
        
        return (modifiers_combined, found_head)
    
    
# def get_head_and_mod_token(ingredient):
#     modifiers_combined, ingredient_head  = analyze_grammar(ingredient)
#     return nlp_score(ingredient_head), nlp_score(modifiers_combined)



# def check_similarity(baseIngredient, ingredientPool, thresholdHead=0.8, thresholdMod=0.8,debug=False):
#     # get 
#     baseTokenHead, baseTokenModifiersCombined = get_head_and_mod_token(baseIngredient)
#     # if debug:
#     #     print(f"--- Ähnlichkeit zu '{baseIngredient}' Head: {baseTokenHead}, Mod: {baseTokenModifiersCombined} ---")
    
#     results = []
#     for comparableIngredient in ingredientPool:

#         comparableTokenHead, comparableTokenModifiersCombined = get_head_and_mod_token(comparableIngredient)
        
#         # Hier passiert die Magie: Cosine Similarity (0.0 bis 1.0)
#         scoreHead = baseTokenHead.similarity(comparableTokenHead)
#         scoreModifiersCombined = -10
#         if baseTokenModifiersCombined.text and comparableTokenModifiersCombined.text:
#             scoreModifiersCombined = baseTokenModifiersCombined.similarity(comparableTokenModifiersCombined)
        
#         if debug:
#             print(f"{comparableIngredient:<30} | Head: {scoreHead:.4f} | Mod: {scoreModifiersCombined:.4f}")

#         if (scoreHead > thresholdHead and not scoreHead > 1.0) and (scoreModifiersCombined > thresholdMod or scoreModifiersCombined == -10):
#             results.append((comparableIngredient, scoreHead, scoreModifiersCombined))   
    
#     # Sortieren nach höchster Ähnlichkeit
#     results.sort(key=lambda x: x[1], reverse=True)
#     if debug:
#         for comparableIngredient, scoreHead, scoreModifiersCombined in results:
#             # Balken zur Visualisierung
#             barHead = "█" * int(scoreHead * 10) 
#             barModifiersCombined = "█" * int(scoreModifiersCombined * 10) 
#             print(f"{comparableIngredient:<20} | {scoreHead:.4f} {barHead}  {scoreModifiersCombined:.4f} {barModifiersCombined}")
#     return {baseIngredient : results}


In [ ]:
# --- Schneller IngredientPool-Index für NLP-Vergleiche ---
class IngredientTokenCache:
    def __init__(self, ingredient_pool):
        """Berechnet einmalig alle NLP-Tokens für schnellere Vergleiche"""
        print(f"Initialisiere Token-Cache für {len(ingredient_pool)} Zutaten...")
        self.token_dict = {}
        for idx, ingredient in enumerate(ingredient_pool):
            if idx % 100 == 0:
                print(f"  Fortschritt: {idx}/{len(ingredient_pool)}")
            self.token_dict[ingredient] = self.get_head_and_mod_token(ingredient)
        print("Token-Cache bereit!")
    
    def get_head_and_mod_token(self, ingredient):
        """Konvertiert eine Zutat in NLP-Tokens (Head + Modifiers)"""
        modifiers_combined, ingredient_head = analyze_grammar(ingredient)
        return nlp_score(ingredient_head), nlp_score(modifiers_combined)
    
    def check_similarity(self, baseIngredient, thresholdHead=0.8, thresholdMod=0.8, debug=False):
        """
        Vergleicht baseIngredient mit allen gecachten Zutaten.
        Returns: { baseIngredient: [(ingredient, scoreHead, scoreMod), ...] }
        """
        baseTokenHead, baseTokenModifiersCombined = self.get_head_and_mod_token(baseIngredient)
        results = []
        
        for comparableIngredient, (comparableTokenHead, comparableTokenModifiersCombined) in self.token_dict.items():
            scoreHead = baseTokenHead.similarity(comparableTokenHead)
            scoreModifiersCombined = -10
            
            if baseTokenModifiersCombined.text and comparableTokenModifiersCombined.text:
                scoreModifiersCombined = baseTokenModifiersCombined.similarity(comparableTokenModifiersCombined)
            
            if debug:
                print(f"{comparableIngredient:<30} | Head: {scoreHead:.4f} | Mod: {scoreModifiersCombined:.4f}")
            
            if (scoreHead > thresholdHead and not scoreHead > 1.0) and (scoreModifiersCombined > thresholdMod or scoreModifiersCombined == -10):
                results.append((comparableIngredient, scoreHead, scoreModifiersCombined))
        
        results.sort(key=lambda x: x[1], reverse=True)
        
        if debug:
            for comparableIngredient, scoreHead, scoreModifiersCombined in results:
                barHead = "█" * int(scoreHead * 10)
                barModifiersCombined = "█" * int(scoreModifiersCombined * 10)
                print(f"{comparableIngredient:<20} | {scoreHead:.4f} {barHead}  {scoreModifiersCombined:.4f} {barModifiersCombined}")
        
        return {baseIngredient: results}

# Beispielnutzung:
# cache = IngredientTokenCache(ALL_INGREDIENTS)
# result = cache.check_similarity("Tomato", thresholdHead=0.9)


In [41]:
import json
from datetime import datetime

def write_JSON_similar_ingredients_fast():
    """
    Erstellt eine JSON-Datei mit ähnlichen Zutaten für alle Zutaten.
    Nutzt IngredientTokenCache für schnellere Verarbeitung.
    Format: { "Zutat": ["Ähnliche1", "Ähnliche2", ...] }
    Dateiname: ingredient_similarity_cache_DD-MM-YYYY-HH-MM.json
    """
    
    # Cache einmalig initialisieren (das dauert, spart aber enorm Zeit danach)
    cache = IngredientTokenCache(ALL_INGREDIENTS)
    
    similarity_data = {}
    
    print(f"\nVerarbeite {len(ALL_INGREDIENTS)} Zutaten...")
    
    for index, ingredient in enumerate(ALL_INGREDIENTS):
        if index % 50 == 0:
            print(f"Fortschritt: {index}/{len(ALL_INGREDIENTS)}")
        
        # Ähnliche Zutaten finden (jetzt viel schneller!)
        result = cache.check_similarity(ingredient, thresholdHead=0.9, thresholdMod=0.8, debug=False)
        
        # Nur die Zutaten-Namen extrahieren (ohne Scores)
        if ingredient in result and result[ingredient]:
            similarity_data[ingredient] = list(result[ingredient])
    
    # Dateiname mit Datum erstellen
    timestamp = datetime.now().strftime("%d-%m-%Y-%H-%M")
    output_file = f"ingredient_similarity_cache_{timestamp}.json"
    
    # JSON speichern (nur das Dictionary)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(similarity_data, f, indent=2, ensure_ascii=False)
    
    print(f"\nFertig! Datei gespeichert: {output_file}")
    print(f"Zutaten mit Matches: {len(similarity_data)}")
    
    return similarity_data

# Funktion ausführen (jetzt mit Cache - viel schneller!)
similarity_results = write_JSON_similar_ingredients_fast()


Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877
  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!

Verarbeite 877 Zutaten...
Fortschritt: 0/877


C:\Users\maxi9\AppData\Local\Temp\ipykernel_30116\2133068240.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)
C:\Users\maxi9\AppData\Local\Temp\ipykernel_30116\2133068240.py:31: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreModifiersCombined = baseTokenModifiersCombined.similarity(comparableTokenModifiersCombined)


Fortschritt: 50/877
Fortschritt: 100/877
Fortschritt: 150/877
Fortschritt: 200/877
Fortschritt: 250/877
Fortschritt: 300/877
Fortschritt: 350/877
Fortschritt: 400/877
Fortschritt: 450/877
Fortschritt: 500/877
Fortschritt: 550/877
Fortschritt: 600/877
Fortschritt: 650/877
Fortschritt: 700/877
Fortschritt: 750/877
Fortschritt: 800/877
Fortschritt: 850/877

Fertig! Datei gespeichert: ingredient_similarity_cache_30-01-2026-13-48.json
Zutaten mit Matches: 877


In [37]:
def validate_similarity_cache(json_file, all_ingredients_list):
    """
    Validiert die erstellte JSON-Datei:
    - Prüft, ob jede Zutat einen Key hat
    - Prüft, ob die Listen nicht leer sind
    """
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            cache_data = json.load(f)
    except Exception as e:
        print(f"Fehler beim Lesen der Datei: {e}")
        return False
    
    print(f"Validiere {json_file}...")
    print(f"Zutaten in ALL_INGREDIENTS: {len(all_ingredients_list)}")
    print(f"Keys in JSON: {len(cache_data)}")
    
    missing_keys = []
    empty_lists = []
    
    # Prüfe, ob alle Zutaten einen Key haben
    for ingredient in all_ingredients_list:
        if ingredient not in cache_data:
            missing_keys.append(ingredient)
        elif not cache_data[ingredient]:  # Liste ist leer
            empty_lists.append(ingredient)
    
    # Ausgabe der Ergebnisse
    print("\n--- VALIDIERUNGSERGEBNISSE ---")
    
    if missing_keys:
        print(f"\n❌ FEHLER: {len(missing_keys)} Zutaten ohne Key:")
        for ing in missing_keys[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(missing_keys) > 10:
            print(f"  ... und {len(missing_keys) - 10} mehr")
    else:
        print("✅ Alle Zutaten haben einen Key")
    
    if empty_lists:
        print(f"\n⚠️ WARNUNG: {len(empty_lists)} Zutaten haben leere Listen:")
        for ing in empty_lists[:10]:  # Zeige nur die ersten 10
            print(f"  - {ing}")
        if len(empty_lists) > 10:
            print(f"  ... und {len(empty_lists) - 10} mehr")
    else:
        print("✅ Keine leeren Listen gefunden")
    
    # Zusammenfassung
    success = len(missing_keys) == 0 and len(empty_lists) == 0
    print(f"\n{'✅ VALIDIERUNG ERFOLGREICH' if success else '❌ VALIDIERUNG FEHLGESCHLAGEN'}")
    
    return success

# Beispielnutzung (nach Ausführung von write_JSON_similar_ingredients_fast()):
validate_similarity_cache("ingredient_similarity_cache_30-01-2026-13-31.json", ALL_INGREDIENTS)


Validiere ingredient_similarity_cache_30-01-2026-13-31.json...
Zutaten in ALL_INGREDIENTS: 877
Keys in JSON: 877

--- VALIDIERUNGSERGEBNISSE ---
✅ Alle Zutaten haben einen Key
✅ Keine leeren Listen gefunden

✅ VALIDIERUNG ERFOLGREICH


True

In [33]:
# --- TEST mit Cache ---
cache = IngredientTokenCache(ALL_INGREDIENTS)
result = cache.check_similarity("Cucumber", thresholdHead=0.8, debug=False)
print(result)


Initialisiere Token-Cache für 877 Zutaten...
  Fortschritt: 0/877
  Fortschritt: 100/877
  Fortschritt: 200/877
  Fortschritt: 300/877
  Fortschritt: 400/877
  Fortschritt: 500/877
  Fortschritt: 600/877
  Fortschritt: 700/877
  Fortschritt: 800/877
Token-Cache bereit!
{'Cucumber': [('Cucumber', 1.0, -10), ('Persian Cucumber', 1.0, -10)]}


C:\Users\maxi9\AppData\Local\Temp\ipykernel_30116\2219725144.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)


In [34]:
result = cache.check_similarity("Cucumber", thresholdHead=0.8, debug=False)
print(result)

{'Cucumber': [('Cucumber', 1.0, -10), ('Persian Cucumber', 1.0, -10)]}


d:\GitRepo\PKI_Projekt_Gruppe_B1_4\.venv\lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
C:\Users\maxi9\AppData\Local\Temp\ipykernel_30116\2219725144.py:27: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scoreHead = baseTokenHead.similarity(comparableTokenHead)
